# 03 - Masks: SAM 2.1 Hiera-S
**Cantaloupe flowers - zero-shot pseudo-labels (no ground truth)**

Turns the Grounding DINO pseudo-label boxes into flower masks with SAM 2.1 Hiera-S (facebook/sam2.1-hiera-small), without training. Fallback for SAM 3, which is gated and not yet approved for this account. The cantaloupe pose method (FloPE, arXiv 2503.11692) starts from flower segmentation, so masks are the more useful label.

No training, so no curves. No labels, so no accuracy: the check is whether a box gets the same mask when the photo is mirrored.

Dataset: https://www.kaggle.com/datasets/samirhossain2001/cantaloupe-flower-datasets

**Attach as inputs:** the dataset + the outputs of notebook 00 and of the `gdino` notebook

## 1 - Setup

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'transformers==5.17.0'], check=True)

In [ ]:
import os
import gc
import re
import json
import glob
import time
import random
import hashlib
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

NB_START = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cv2", cv2.__version__,
      "| device", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| cpu cores", os.cpu_count())

## 2 - Configuration

In [ ]:
@dataclass
class CFG:
    INPUT_ROOTS: list = field(default_factory=lambda: os.environ.get("INPUT_ROOTS", '/kaggle/input').split(os.pathsep))
    OUT_DIR: str = '/kaggle/working' if os.path.isdir('/kaggle/working') else "outputs"
    DATA_DIR: str = '/kaggle/input/datasets/samirhossain2001/cantaloupe-flower-datasets/Cantaloupe flower datasets'  # dataset folder on Kaggle, checked first; otherwise the inputs are searched
    MODEL_NAME: str = 'SAM 2.1 Hiera-S'
    TAG: str = 'gdino_sam2'
    CHECKPOINT: str = 'facebook/sam2.1-hiera-small'
    BOX_TAG: str = 'gdino'  # the notebook whose boxes prompt the masks
    MATCH_IOU: float = 0.5
    N_BOOT: int = 1000
    SEED: int = 42


cfg = CFG()
SMOKE = False
cfg

## 3 - Shared helpers

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def find_artifact(fname, required=True):
    for root in cfg.INPUT_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, "**", fname), recursive=True))
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(f"{fname} not found: attach the output of the notebook that writes it as an input.")
    return None


def find_data_root(rel_path):
    if os.path.exists(os.path.join(cfg.DATA_DIR, rel_path)):
        return cfg.DATA_DIR
    for root in cfg.INPUT_ROOTS:
        for dirpath, _, _ in os.walk(root):
            if os.path.exists(os.path.join(dirpath, rel_path)):
                return dirpath
    raise FileNotFoundError(f"{rel_path} not found: attach the dataset as an input.")


def read_rgb(rel_path):
    a = cv2.imread(os.path.join(DATA_ROOT, rel_path), cv2.IMREAD_COLOR)
    assert a is not None, f"unreadable image: {rel_path}"
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB)


def dihedral(a, k):
    """k = 0..3: rotate k x 90 degrees counter-clockwise; k = 4..7: the same, then mirror left-right."""
    a = np.rot90(a, k % 4)
    return np.ascontiguousarray(a[:, ::-1] if k >= 4 else a)


def inv_k(k):
    """Inverse of dihedral(., k): rotations undo with the opposite turn, mirrored ones are their own inverse."""
    return k if k >= 4 else (4 - k) % 4


def dihedral_points(xy, k, w, h):
    xy = np.asarray(xy, dtype=np.float64).reshape(-1, 2).copy()
    for _ in range(k % 4):
        xy = np.stack([xy[:, 1], w - xy[:, 0]], axis=1)
        w, h = h, w
    if k >= 4:
        xy[:, 0] = w - xy[:, 0]
    return xy


def dihedral_boxes(boxes, k, w, h):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes
    p1, p2 = dihedral_points(boxes[:, :2], k, w, h), dihedral_points(boxes[:, 2:], k, w, h)
    return np.concatenate([np.minimum(p1, p2), np.maximum(p1, p2)], axis=1)


def box_area(b):
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    a, b = np.asarray(a, dtype=np.float64).reshape(-1, 4), np.asarray(b, dtype=np.float64).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    return inter / np.maximum(box_area(a)[:, None] + box_area(b)[None, :] - inter, 1e-9)


def match_f1_from_iou(iou, n_a, n_b, thr):
    """F1 of a one-to-one matching at IoU >= thr; nan when both sides are empty."""
    if n_a + n_b == 0:
        return float("nan")
    if n_a == 0 or n_b == 0:
        return 0.0
    r, c = linear_sum_assignment(-iou)
    return 2 * int((iou[r, c] >= thr).sum()) / (n_a + n_b)


def match_f1(a, b, thr):
    return match_f1_from_iou(box_iou(a, b), len(a), len(b), thr)


def boot_ci(values, n_boot, seed=0):
    v = np.asarray([x for x in values if not np.isnan(x)], dtype=np.float64)
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    means = np.random.default_rng(seed).choice(v, (n_boot, len(v))).mean(1)
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def hex_rgb(h):
    return tuple(int(h[i:i + 2], 16) for i in (1, 3, 5))


def tag_rect(img, text, x, y, scale):
    """Rectangle of a text tag whose bottom-left corner is (x, y), kept inside the image."""
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    x = int(min(max(x, 0), img.shape[1] - tw - 6))
    y = int(min(max(y, th + base + 4), img.shape[0]))
    return x, y - th - base - 4, x + tw + 6, y, base


def put_tag(img, text, x, y, rgb, scale):
    x1, y1, x2, y2, base = tag_rect(img, text, x, y, scale)
    cv2.rectangle(img, (x1, y1), (x2, y2), rgb, -1)
    cv2.putText(img, text, (x1 + 3, y2 - base - 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)
    return x1, y1, x2, y2


def draw_boxes(img, boxes, scores=None, color="#2a78d6", numbers=False, thick=None):
    """Thick box with a dark outline, so it shows on leaves and on bright sky; label on a filled tag at the first
    corner of the box where it does not cover an earlier tag."""
    out = img.copy()
    rgb = hex_rgb(color)
    side = max(out.shape[:2])
    t = thick or max(3, round(side / 160))
    scale = max(0.6, side / 900)
    placed = []
    for i, b in enumerate(np.asarray(boxes).reshape(-1, 4).astype(int)):
        p1, p2 = (int(b[0]), int(b[1])), (int(b[2]), int(b[3]))
        cv2.rectangle(out, p1, p2, (20, 20, 20), t + 3, cv2.LINE_AA)
        cv2.rectangle(out, p1, p2, rgb, t, cv2.LINE_AA)
        label = str(i) if numbers else (f"{scores[i]:.2f}" if scores is not None else "")
        if not label:
            continue
        (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
        h = th + base + 4
        o = (t + 3) // 2
        corners = [(p1[0] - o, p1[1] - o), (p1[0] - o, p2[1] + o + h), (p2[0] - tw - 6 + o, p1[1] - o),
                   (p2[0] - tw - 6 + o, p2[1] + o + h), (p1[0] + o, p1[1] + o + h)]
        free = [c for c in corners if not any(
            r[0] < q[2] and q[0] < r[2] and r[1] < q[3] and q[1] < r[3] for r in [tag_rect(out, label, *c, scale)[:4]] for q in placed)]
        placed.append(put_tag(out, label, *(free[0] if free else corners[0]), rgb, scale))
    return out


def finish(fig, title):
    """Title above the panels (also for tall figures), then show."""
    h = fig.get_figheight()
    fig.suptitle(title, y=1 - 0.12 / h, va="top")
    plt.tight_layout(rect=(0, 0, 1, 1 - 0.5 / h))
    plt.show()


def save_json(obj, name):
    path = os.path.join(cfg.OUT_DIR, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)
    return path


os.makedirs(cfg.OUT_DIR, exist_ok=True)
seed_everything(cfg.SEED)

probe = np.zeros((40, 60), np.uint8)
probe[5:12, 30:50] = 1
for k in range(8):
    ys, xs = np.nonzero(dihedral(probe, k))
    assert np.allclose(dihedral_boxes([[30, 5, 50, 12]], k, 60, 40)[0], [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]), k
    assert np.array_equal(dihedral(dihedral(probe, k), inv_k(k)), probe), k
print("box transform matches the image transform, and each inverse undoes it, for all 8 flips / rotations")

## 4 - Load the split and the boxes

In [ ]:
split_meta = json.load(open(find_artifact("split.json")))
ITEMS = pd.DataFrame([dict(it, split=s) for s, its in split_meta["items"].items() for it in its])
DATA_ROOT = find_data_root(ITEMS.img.iloc[0])
QUAL = split_meta["qual_sources"]
if SMOKE:
    pick = []
    for s in ("train", "validation", "test"):
        part = ITEMS[ITEMS.split == s]
        pick += list(part.groupby("source").size().sort_values(ascending=False, kind="stable").index[:3])
    pick += [q for q in QUAL if q in set(ITEMS.source)][:2]
    ITEMS = ITEMS[ITEMS.source.isin(pick)].reset_index(drop=True)
    QUAL = [q for q in QUAL if q in set(ITEMS.source)][:2]
BY_ID = ITEMS.set_index("id", drop=False)
print(f"split {split_meta['split_id']} | images per split:", ITEMS.split.value_counts().to_dict(),
      "| multi-copy sources:", {s: int((g.groupby("source").size() > 1).sum()) for s, g in ITEMS.groupby("split")})

In [ ]:
pseudo = json.load(open(find_artifact(f"pseudo_{cfg.BOX_TAG}.json")))
assert pseudo["info"]["split_id"] == split_meta["split_id"], "notebook 02 ran on another 00 output: rerun it"
assert pseudo["info"].get("propagated"), "notebook 02 output is from an older version: rerun 02 with the current notebook"
img_of = {im["id"]: im["item_id"] for im in pseudo["images"]}
BOXES = {i: [] for i in ITEMS.id}
SCORES = {i: [] for i in ITEMS.id}
for a in pseudo["annotations"]:
    i = img_of[a["image_id"]]
    if i in BOXES:
        x, y, w, h = a["bbox"]
        BOXES[i].append([x, y, x + w, y + h])
        SCORES[i].append(a["score"])
BOXES = {i: np.array(b, dtype=np.float64).reshape(-1, 4) for i, b in BOXES.items()}
print(f"boxes from {pseudo['info']['model']} (prompt '{pseudo['info']['prompt']}', threshold {pseudo['info']['threshold']}): "
      f"{sum(len(b) for b in BOXES.values())} on {len(BOXES)} images")

## 5 - Model
Each box is one prompt; SAM 2.1 returns one mask per box.

In [ ]:
from transformers import Sam2Processor, Sam2Model


def autocast():
    return torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda")


processor = Sam2Processor.from_pretrained(cfg.CHECKPOINT)
model = Sam2Model.from_pretrained(cfg.CHECKPOINT).to(device).eval()


@torch.no_grad()
def segment(img, boxes):
    h, w = img.shape[:2]
    if len(boxes) == 0:
        return np.zeros((0, h, w), dtype=bool)
    inputs = processor(images=img, input_boxes=[boxes.tolist()], return_tensors="pt").to(device)
    with autocast():
        out = model(**inputs, multimask_output=False)
    masks = processor.post_process_masks(out.pred_masks.float().cpu(), inputs["original_sizes"].cpu())[0]
    return masks[:, 0].numpy().astype(bool)


print(f"{cfg.MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters")

## 6 - Masks for every image
SAM 2.1 segments the colour reference of each photo; its grayscale / flipped copies get the same masks through the transforms from 00, matching the copied boxes.

In [ ]:
t0 = time.time()
MASKS = {}
own = ITEMS[ITEMS.is_ref | ~ITEMS.aligned]
for i in tqdm(list(own.id), desc="masks (reference copies)"):
    MASKS[i] = segment(read_rgb(BY_ID.img[i]), BOXES[i])
REF_OF = ITEMS[ITEMS.is_ref].set_index("source").id
for it in ITEMS.itertuples():
    if it.id not in MASKS:
        m = MASKS[REF_OF[it.source]]
        MASKS[it.id] = np.stack([dihedral(x, inv_k(it.tf)) for x in m]) if len(m) else np.zeros((0, it.height, it.width), dtype=bool)
        assert len(MASKS[it.id]) == len(BOXES[it.id]), it.id
RUN_S = time.time() - t0
fill = np.concatenate([m.reshape(len(m), -1).sum(1) / np.maximum(box_area(BOXES[i]), 1) for i, m in MASKS.items() if len(m)] or [np.zeros(0)])
tight = []
for i, m in MASKS.items():
    for k in range(len(m)):
        ys, xs = np.nonzero(m[k])
        tight.append(box_iou([[xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]], BOXES[i][k:k + 1])[0, 0] if len(xs) else 0.0)
tight = np.array(tight)
print(f"{sum(len(MASKS[i]) for i in own.id)} masks on {len(own)} reference photos in {RUN_S / 60:.1f} min | "
      f"copied to {len(ITEMS) - len(own)} Roboflow copies | empty masks {int((fill == 0).sum())}")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(fill, bins=30, color="#2a78d6")
axes[0].set_title("Mask area / box area")
axes[1].hist(tight, bins=30, color="#1baf7a")
axes[1].set_title("IoU of the mask's tight box with the input box")
plt.tight_layout()
plt.show()

## 7 - Label-free evaluation
Each photo is segmented again mirrored, with mirrored boxes; masks are mapped back and compared box by box.

In [ ]:
def mask_iou(a, b):
    a, b = a.reshape(len(a), -1).astype(np.float32), b.reshape(len(b), -1).astype(np.float32)
    inter = a @ b.T
    return inter / np.maximum(a.sum(1)[:, None] + b.sum(1)[None, :] - inter, 1e-9)


RESULTS = {}
for s in ("validation", "test"):
    refs = [i for i in ITEMS[(ITEMS.split == s) & ITEMS.is_ref].id if len(BOXES[i])]
    per_photo = {}
    for i in tqdm(refs, desc=f"mirrored masks | {s}", leave=False):
        img = read_rgb(BY_ID.img[i])
        h, w = img.shape[:2]
        mm = segment(dihedral(img, 4), dihedral_boxes(BOXES[i], 4, w, h))
        back = np.stack([dihedral(m, 4) for m in mm])
        per_photo[i] = float(np.diag(mask_iou(MASKS[i], back)).mean())
    mean, lo, hi = boot_ci(list(per_photo.values()), cfg.N_BOOT, cfg.SEED)
    RESULTS[s] = {"mask_mirror_iou": mean, "mirror_ci": [lo, hi], "n_photos": len(per_photo), "per_photo": per_photo}
    print(f"{s:10s} mask IoU between the original and the mirrored pass (same box): {mean:.3f} [95% CI {lo:.3f}, {hi:.3f}] "
          f"over {len(per_photo)} photos")
print("Label-free: this checks that a box gets the same mask when the photo is mirrored, not that the mask is correct.")

## 8 - Sample predictions

In [ ]:
def overlay(img, masks, color="#eb6834"):
    out = img.copy().astype(np.float32)
    rgb = np.array(hex_rgb(color), dtype=np.float32)
    for m in masks:
        out[m] = 0.45 * out[m] + 0.55 * rgb
        cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, cnts, -1, (20.0, 20.0, 20.0), 6, cv2.LINE_AA)
        cv2.drawContours(out, cnts, -1, tuple(float(v) for v in rgb), 3, cv2.LINE_AA)
    return out.clip(0, 255).astype(np.uint8)


refs = [ITEMS[(ITEMS.source == s) & ITEMS.is_ref].id.iloc[0] for s in QUAL if (ITEMS.source == s).any()][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10), squeeze=False)
for a, i in zip(axes.ravel(), refs):
    a.imshow(draw_boxes(overlay(read_rgb(BY_ID.img[i]), MASKS[i]), BOXES[i], color="#2a78d6"))
    a.set_title(f"{BY_ID.source[i]} | {len(MASKS[i])} masks", fontsize=9)
for a in axes.ravel():
    a.axis("off")
plt.suptitle("SAM 2.1 masks (orange) from Grounding DINO boxes (blue), fixed test photos")
plt.tight_layout()
plt.show()

## 9 - Saved outputs
COCO json with polygons and YOLO segmentation txt (class 0 = flower).

In [ ]:
def polygons(m):
    cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c.reshape(-1).astype(float).tolist() for c in sorted(cnts, key=cv2.contourArea, reverse=True) if len(c) >= 3]


images, anns = [], []
for k, it in enumerate(ITEMS.sort_values("id").itertuples(), 1):
    images.append({"id": k, "file_name": it.img, "item_id": it.id, "split": it.split, "width": int(it.width), "height": int(it.height)})
    d = os.path.join(cfg.OUT_DIR, f"yolo_{cfg.TAG}", "labels", it.split)
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, os.path.splitext(os.path.basename(it.img))[0] + ".txt"), "w") as f:
        for m, b, s in zip(MASKS[it.id], BOXES[it.id], SCORES[it.id]):
            poly = polygons(m)
            if not poly:
                continue
            x1, y1, x2, y2 = [float(v) for v in b]
            anns.append({"id": len(anns) + 1, "image_id": k, "category_id": 1, "bbox": [x1, y1, x2 - x1, y2 - y1],
                         "area": float(m.sum()), "score": float(s), "segmentation": poly, "iscrowd": 0})
            xy = np.array(poly[0]).reshape(-1, 2) / [it.width, it.height]
            f.write("0 " + " ".join(f"{v:.6f}" for v in xy.ravel()) + "\n")
with open(os.path.join(cfg.OUT_DIR, f"yolo_{cfg.TAG}", "data.yaml"), "w") as f:
    f.write("# YOLO segmentation labels only; images are in the dataset. Pseudo-labels, not ground truth.\nnames:\n  0: flower\n")
save_json({"info": {"tag": cfg.TAG, "model": cfg.MODEL_NAME, "boxes_from": cfg.BOX_TAG, "split_id": split_meta["split_id"],
                    "note": "zero-shot pseudo-labels, not ground truth"},
           "images": images, "annotations": anns, "categories": [{"id": 1, "name": "flower"}]}, f"pseudo_{cfg.TAG}.json")
params = sum(p.numel() for p in model.parameters())
metrics = {"tag": cfg.TAG, "model": cfg.MODEL_NAME, "checkpoint": cfg.CHECKPOINT, "boxes_from": cfg.BOX_TAG,
           "split_id": split_meta["split_id"], "task": "zero_shot_masks", "main_metric": "mask_mirror_iou",
           "higher_better": True, "bounded": True, "label_free": True,
           "validation": {k: v for k, v in RESULTS["validation"].items() if k != "per_photo"},
           "test": {k: v for k, v in RESULTS["test"].items() if k != "per_photo"},
           "mask_fill_median": float(np.median(fill)) if len(fill) else None,
           "tight_box_iou_median": float(np.median(tight)) if len(tight) else None,
           "params_m": params / 1e6, "images_per_s": len(ITEMS) / max(RUN_S, 1e-9),
           "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None, "smoke": SMOKE}
save_json(metrics, f"metrics_{cfg.TAG}.json")
save_json({s: RESULTS[s]["per_photo"] for s in RESULTS}, f"per_item_{cfg.TAG}.json")
print(f"saved pseudo_{cfg.TAG}.json ({len(anns)} masks), yolo_{cfg.TAG}/, metrics_{cfg.TAG}.json | total {(time.time() - NB_START) / 60:.1f} min")